# Classification: predicting *categories*, not numbers

So far both notebooks predicted a number along a line/curve (**regression**). Classification predicts **which class** something belongs to.

Instead of a continuous loss we need:
- a **different output layer**: 1 raw score per example (a *logit*), converted to a probability with **sigmoid**
- a **different loss**: `nn.BCEWithLogitsLoss()` (binary cross-entropy) - `nn.L1Loss()` makes no sense for yes/no
- a **different metric**: accuracy - loss is for the optimizer's benefit; accuracy is for *ours*

Toy problem: **two circles**, inner vs outer - impossible to separate with a straight line, so this forces us to reuse exactly what the nonlinear notebook taught: layers + ReLU.

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch import nn
from sklearn.datasets import make_circles

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. Data: `make_circles`

`make_circles` gives 2-D points (x1, x2) labelled 0 or 1 depending on which ring they sit in.

In [3]:
n_samples = 1200
X, y = make_circles(n_samples=n_samples, noise=0.06, random_state=42)

# to torch tensors: features float32 (2 per sample), labels float32 (1 per sample)
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float).unsqueeze(dim=1)

X.shape, y.shape, X[:5], y[:5].squeeze()

In [4]:
# 80:20 split, same as always
train_split = int(0.8 * n_samples)
X_train, y_train = X[:train_split].to(device), y[:train_split].to(device)
X_test, y_test = X[train_split:].to(device), y[train_split:].to(device)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [5]:
def plot_circles(X, y, predictions=None):
    """Scatter the 2 classes; optionally mark wrong predictions in red."""
    plt.figure(figsize=(7, 7))
    plt.scatter(X[y.squeeze() == 0, 0].cpu(), X[y.squeeze() == 0, 1].cpu(),
                c="b", s=15, label="class 0 (inner)")
    plt.scatter(X[y.squeeze() == 1, 0].cpu(), X[y.squeeze() == 1, 1].cpu(),
                c="orange", s=15, label="class 1 (outer)")
    if predictions is not None:
        wrong = (predictions != y).squeeze()
        plt.scatter(X[wrong, 0].cpu(), X[wrong, 1].cpu(),
                    c="red", marker="x", s=40, label="wrong predictions")
    plt.legend()
    plt.show()

plot_circles(X.cpu(), y.cpu())

## 2. Model: same architecture as the nonlinear notebook

The input is now **2 features** instead of 1, and the output is 1 raw score (logit) per example. ReLU in the middle, exactly as before.

In [7]:
torch.manual_seed(42)

class CircleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=2, out_features=16)
        self.layer2 = nn.Linear(in_features=16, out_features=16)
        self.layer3 = nn.Linear(in_features=16, out_features=1)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layer3(self.relu(self.layer2(self.relu(self.layer1(x)))))

model = CircleModel().to(device)
model

## 3. Loss and metric

- `nn.BCEWithLogitsLoss()` = sigmoid + binary cross-entropy fused. "WithLogits" matters: it is numerically stable to feed it the *raw* model output, not the sigmoid-ed one.
- accuracy: fraction of predictions that match the labels. We compare `torch.round(torch.sigmoid(logits))` to `y`.

In [9]:
loss_fn = nn.BCEWithLogitsLoss()

def accuracy_fn(y_true, y_pred):
    """Fraction correct, as a percentage."""
    correct = torch.eq(y_true.squeeze(), y_pred.squeeze()).sum().item()
    return correct / len(y_true) * 100

### Why logits, not probabilities?

The model's raw output is any real number (-3.2, 0.7, ...). Two conversions appear constantly:
- `sigmoid(logits)` -> probability in [0, 1]
- `round(...)` or `> 0.5` -> hard 0/1 prediction

Training uses raw logits (the loss does the sigmoid internally, stably); evaluation uses the converted values.

## 4. Training

Same 5-step loop. Note the extra step: convert logits -> predictions only when computing accuracy.

In [12]:
torch.manual_seed(42)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)
epochs = 500

loss_history, acc_history = [], []

for epoch in range(epochs):
    model.train()

    logits = model(X_train)                      # 1. forward pass (raw scores)
    loss = loss_fn(logits, y_train)              # 2. compute loss (on logits!)
    optimizer.zero_grad()                        # 3. prep for backward pass
    loss.backward()                              # 4. backpropagation
    optimizer.step()                             # 5. gradient descent

    with torch.inference_mode():                 # metrics don't need gradients
        train_logits = model(X_train)
        train_preds = torch.round(torch.sigmoid(train_logits))
        acc = accuracy_fn(y_train, train_preds)

    loss_history.append(loss.item())
    acc_history.append(acc)

print(f"epoch: {epoch} | loss: {loss.item():.4f} | accuracy: {acc:.2f}%")

epoch: 499 | loss: 0.0838 | accuracy: 96.67%


In [13]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(loss_history)
plt.title("Loss (BCEWithLogits)")
plt.xlabel("epoch")
plt.subplot(1, 2, 2)
plt.plot(acc_history)
plt.title("Accuracy (%)")
plt.xlabel("epoch")
plt.show()

## 5. Evaluation

In [15]:
model.eval()
with torch.inference_mode():
    test_logits = model(X_test)
    test_preds = torch.round(torch.sigmoid(test_logits))
    test_loss = loss_fn(test_logits, y_test)
    test_acc = accuracy_fn(y_test, test_preds)

print(f"test loss: {test_loss.item():.4f} | test accuracy: {test_acc:.2f}%")

test loss: 0.1507 | test accuracy: 95.00%


In [16]:
# where did it get it wrong?
plot_circles(X_test.cpu(), y_test.cpu(), predictions=test_preds.cpu())

## 6. Why a straight line model *cannot* do this

The classes are two rings - no straight line can separate them. To see it, train the same data on a model with **no ReLU**: it is linear, so its decision boundary is a line, and accuracy stalls near ~50% (a coin flip).

In [18]:
torch.manual_seed(42)

class LinearOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.layer2 = nn.Linear(16, 16)
        self.layer3 = nn.Linear(16, 1)
        # no ReLU anywhere

    def forward(self, x):
        return self.layer3(self.layer2(self.layer1(x)))

model_linear = LinearOnly().to(device)
optimizer = torch.optim.Adam(model_linear.parameters(), lr=0.01)
epochs = 500

for epoch in range(epochs):
    model_linear.train()
    loss = loss_fn(model_linear(X_train), y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model_linear.eval()
with torch.inference_mode():
    preds = torch.round(torch.sigmoid(model_linear(X_test)))
print(f"linear-only model accuracy: {accuracy_fn(y_test, preds):.2f}%")

linear-only model accuracy: 37.50%


~50-60%, versus ~95%+ for the model with ReLU. Removing one line of nonlinearity costs most of the accuracy - that is how central nonlinearity is.

## 7. Saving the model

Same routine as the regression notebook - save the `state_dict()` and rebuild from it.

In [21]:
from pathlib import Path

MODEL_PATH = Path("models")
MODEL_PATH.mkdir(exist_ok=True)
torch.save(obj=model.state_dict(), f=MODEL_PATH / "circle_model.pth")
print("saved:", MODEL_PATH / "circle_model.pth")

saved: models/circle_model.pth


In [22]:
loaded_model = CircleModel().to(device)
loaded_model.load_state_dict(torch.load(MODEL_PATH / "circle_model.pth"))
loaded_model.eval()

with torch.inference_mode():
    reloaded_preds = torch.round(torch.sigmoid(loaded_model(X_test)))
print(f"reloaded accuracy: {accuracy_fn(y_test, reloaded_preds):.2f}%")

reloaded accuracy: 95.00%


## 8. What we learned

1. **classification = categories**: output is 1 logit per example, converted with sigmoid (probabilities) and thresholding (hard guesses)
2. **`nn.BCEWithLogitsLoss`** for binary classification - feed it raw logits; convert before computing accuracy
3. **accuracy** is a metric, not a loss - it lives in the evaluation mindset
4. **nonlinearity is load-bearing**: without ReLU the same architecture scores ~50% on circles, with it ~95%
5. `make_circles` is the standard toy for "linear separation is impossible"

Next step: multi-class classification (3+ classes needs softmax and `nn.CrossEntropyLoss`), then computer vision with `Dataset`/`DataLoader` and convolutions on FashionMNIST.